# Очистка данных: Contacts

In [17]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


import help_130625_dam as h

# Пути к данным
DATA_PATH = os.path.join('..', 'Sources', 'Contacts (Done).xlsx')
OUT_PATH  = os.path.join('..', 'data', 'cleaned', 'contacts_clean.pkl')

## Загрузка и первичный осмотр

In [18]:
df = pd.read_excel(DATA_PATH)

# Переименование столбцов в snake_case
df.columns = [col.lower().replace(' ', '_') for col in df.columns]

print(f'Форма: {df.shape}')
print(f'Столбцы: {list(df.columns)}')
n_before = df.shape[0]

df.head()

Форма: (18548, 4)
Столбцы: ['id', 'contact_owner_name', 'created_time', 'modified_time']


,id,contact_owner_name,created_time,modified_time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17
3,5805028000000907006,Bob Brown,03.07.2023 05:44,29.12.2023 15:20
4,5805028000000939010,Nina Scott,04.07.2023 10:11,16.04.2024 16:14


In [19]:
# Полные дубликаты (все столбцы)
full_dupes = df.duplicated().sum()
print(f'Полных дубликатов: {full_dupes}')

# Дубликаты по id
id_dupes = df.duplicated(subset='id').sum()
print(f'Дубликатов по id: {id_dupes}')

Полных дубликатов: 0
Дубликатов по id: 0


После нескольких итераций и разбора источника пришел к выводу:
Пропусков нет. Дубликатов нет. Важно ИД зафиксировать как int64. Есть один менеджер "False" у которого у которого один клиент. В сделках у этого клиента другой менеджер. Значит этот менеджер техническая ошибка. Заменяем менеджера на Jane Smith, так как она первая в звонках у этого клиента

In [20]:
df.loc[df['id'] == 5805028000008772190, 'contact_owner_name'] = 'Jane Smith'

In [21]:
h.descr_df(df, include='all', show_stats=False, show_sample_rows=True)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3
0,id,int64,18548,0,18548,5805028000000645014,5805028000000872003,5805028000000889001
1,contact_owner_name,object,18548,0,27,Rachel White,Charlie Davis,Bob Brown
2,created_time,str,18548,0,17921,27.06.2023 11:28,03.07.2023 11:31,02.07.2023 22:37
3,modified_time,str,18548,0,16580,22.12.2023 13:34,21.05.2024 10:23,21.12.2023 13:17


## Типы данных: даты

In [22]:
DATE_COLS = ['created_time', 'modified_time']

for col in DATE_COLS:
    # Пробуем автоопределение формата (dayfirst=True для DD.MM.YYYY)
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

# Проверяем результат
print('Типы после парсинга:')
print(df[DATE_COLS].dtypes)
print()

# Считаем NaT (не распарсились)
for col in DATE_COLS:
    nat_count = df[col].isna().sum()
    print(f'{col}: NaT = {nat_count} ({nat_count/len(df)*100:.2f}%)')


Типы после парсинга:
created_time     datetime64[us]
modified_time    datetime64[us]
dtype: object

created_time: NaT = 0 (0.00%)
modified_time: NaT = 0 (0.00%)


In [23]:
# Диапазон дат
for col in DATE_COLS:
    print(f'{col}: {df[col].min()}  →  {df[col].max()}')

created_time: 2023-06-27 11:28:00  →  2024-06-21 15:30:00
modified_time: 2023-07-06 10:54:00  →  2024-06-21 15:32:00


In [24]:
# Проверка логики: modified_time не должна быть раньше created_time
anomalies = df[df['modified_time'] < df['created_time']]
print(f'Строк где modified < created: {len(anomalies)}')
if len(anomalies) > 0:
    display(anomalies.head(10))


Строк где modified < created: 0


In [25]:
# Преобразование в категориальный тип для оптимизации
df['contact_owner_name'] = df['contact_owner_name'].astype('category')

owner_counts = df['contact_owner_name'].value_counts()
print(f'Уникальных менеджеров: {owner_counts.shape[0]}')

Уникальных менеджеров: 27


## Итоговый осмотр

In [27]:
print('Типы данных финального датасета:')
print(df.dtypes)
print()
df.head()

Типы данных финального датасета:
id                             int64
contact_owner_name          category
created_time          datetime64[us]
modified_time         datetime64[us]
dtype: object



,id,contact_owner_name,created_time,modified_time
0,5805028000000645014,Rachel White,2023-06-27 11:28:00,2023-12-22 13:34:00
1,5805028000000872003,Charlie Davis,2023-07-03 11:31:00,2024-05-21 10:23:00
2,5805028000000889001,Bob Brown,2023-07-02 22:37:00,2023-12-21 13:17:00
3,5805028000000907006,Bob Brown,2023-07-03 05:44:00,2023-12-29 15:20:00
4,5805028000000939010,Nina Scott,2023-07-04 10:11:00,2024-04-16 16:14:00


In [28]:
h.descr_df(df, include=['number', 'category', 'datetime64'], show_stats=False, show_sample_rows=False)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений
0,id,int64,18548,0,18548
1,contact_owner_name,category,18548,0,27
2,created_time,datetime64[us],18548,0,17921
3,modified_time,datetime64[us],18548,0,16580


In [31]:
# Обогащение данных: Дата первой оплаты
# Подтягиваем информацию из buyers_info.pkl (сформирован в 03_cleaning_deals)

BUYERS_INFO_PATH = os.path.join('..', 'data', 'cleaned', 'buyers_info.pkl')

if os.path.exists(BUYERS_INFO_PATH):
    buyers_info = pd.read_pickle(BUYERS_INFO_PATH)

    # Приводим ID к типу Int64 для корректного merge
    buyers_info['contact_id'] = buyers_info['contact_id'].astype('Int64')

    # Удаляем столбец если уже был добавлен (защита от повторного запуска ячейки)
    df = df.drop(columns=['first_payment_date'], errors='ignore')

    # Объединяем контакты с данными по оплатам
    df = df.merge(
        buyers_info.rename(columns={'contact_id': 'id'}),
        on='id',
        how='left'
    )

    print(f"Контакты обогащены данными: добавлена дата первой оплаты для {df['first_payment_date'].count()} клиентов.")
else:
    print("Файл buyers_info.pkl не найден. Сначала выполните блокнот 03_cleaning_deals.")


Контакты обогащены данными: добавлена дата первой оплаты для 3212 клиентов.


## Сохранение и итоги

In [32]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_pickle(OUT_PATH)

# Собираем данные для итогов
summary_data = {
    'Метрика': ['Строк исходно', 'Строк после очистки', 'Удалено дубликатов', 'Столбцы дат', 'Пропуски в менеджере'],
    'Значение': [
        n_before, 
        len(df), 
        n_before - len(df), 
        'created_time, modified_time', 
        df['contact_owner_name'].isna().sum()
    ]
}
summary_df = pd.DataFrame(summary_data)

print(f'Сохранено в : {OUT_PATH}')
display(summary_df)

Сохранено в : ..\data\cleaned\contacts_clean.pkl


,Метрика,Значение
0,Строк исходно,18548
1,Строк после очистки,18548
2,Удалено дубликатов,0
3,Столбцы дат,"created_time, modified_time"
4,Пропуски в менеджере,0


## Описание датасета

**Источник:** `Contacts (Done).xlsx` — выгрузка из CRM  
**Назначение:** справочник лидов/клиентов; связывает сделки и звонки с конкретным контактом через `id`

| Столбец | Тип | Описание |
|---|---|---|
| `id` | `int64` | Уникальный ID контакта в CRM (19-значный) |
| `contact_owner_name` | `category` | Менеджер, ответственный за контакт |
| `created_time` | `datetime` | Дата и время создания контакта |
| `year_month` | `period[M]` | месяц/год создания контакта — **основа для когортного анализа** |
| `modified_time` | `datetime` | Дата последнего изменения записи |
| `first_payment_date` | `datetime` | **Обогащённый признак:** дата самого раннего платежа (из `buyers_info.pkl`); NaN = лид без оплаты |

**Объём:** 18 548 контактов, 5 столбцов после обогащения  
**Ключевые связи:**
- `id` → `deals.contact_id` (сделки)
- `id` → `calls.contactid` (звонки)
- `created_time` → `cohort` (месяц первого контакта, используется в `05_data_merging`)
